# SRQS8F7JWA9MZ

In [ ]:
# Imports
from foodcast.imports import *
os.chdir(find_project_root())
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, _, _, _, DATA_DIR_3_7 = DATA_DIR_3_x

# Settings
notebook_settings() 

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
loc_id = 'SRQS8F7JWA9MZ'
df_uncleaned = load_single_restaurant(loc_id)
df_targeted = pd.read_parquet(DATA_DIR_3_7 / f'{loc_id}.parquet')

In [ ]:
# Remapping was manual, but stored in a yaml now to stay organized
remapping_path = Path('scripts') / 'labeling' / 'remapping' / 'loc1_remappings.yaml'
with open(remapping_path, "r", encoding="utf-8") as f:
    remapping = yaml.load(f, Loader=yaml.FullLoader)

# Extract the parts of the yaml to do the programmatic relabeling
df_relabeled = fully_relabel_and_consolidate(
    df                        = df_uncleaned,
    name_changes              = remapping.get("name_changes", {}),
    modification_name_changes = remapping.get("modification_name_changes", []),
    vegan_list                = remapping.get("vegan_list", []),
    vegetarian_list           = remapping.get("vegetarian_list", []),
    meat_list                 = remapping.get("meat_list", []),
    alcohol_list              = remapping.get("alcoholic_drinks", []),
    drinks_list               = remapping.get("non_alcoholic_drinks", []),
    merch                     = remapping.get("merch_list", []),
    rare                      = remapping.get("rare_list", []) + remapping.get("uncommon_list", []),
    unknown                   = remapping.get("unknown_list", []),
    remove_categories         = ["Merch", "Drink", "Rare"],
)

# # Are the labels uniquely specified?
# display(df_relabeled.groupby('item_name')['vegan'].unique()) 

df_relabeled.to_parquet(DATA_DIR_3_1 / (loc_id + '_sales_and_menu.parquet'))

# Consolidate dishes

df_consolidated = rename_items(
    df = df_relabeled,
    name_changes = {
        'Gold Standard - Bacon' : [],
        'Gold Standard - Kale' : ["Vegan Gold Standard - Kale", "Meat Gold Standard - Kale"],
        'Gold Standard - Bacon & Kale' :[],
        'Gold Standard - Impossible' : ['Meat Gold Standard - Impossible', 'Vegan Gold Standard - Impossible'],
        'Beyond Burger' : ['Meat Beyond Burger', 'Vegan Beyond Burger'],
        'Impossible Patty Melt' : ['Meat Impossible Patty Melt', 'Vegan Impossible Patty Melt'],
        'The Alternative' : ['Meat The Alternative', 'Vegetarian The Alternative', 'Impossible The Alternative'],
        'Bottled Pop': df_relabeled.value_counts('item_name').filter(regex='Coca|Crod|Ginger|Soda').index.tolist(),
        'Canned Drinks': df_relabeled.value_counts('item_name').filter(regex='Coco|Mate|Zam|Croi').index.tolist(),
    }
)

# # There should be multiple labels for each item
# display(df_consolidated.groupby('item_name')['vegan'].unique()) 

df_consolidated.to_parquet(DATA_DIR_3_2 / (loc_id + '_sales_and_menu.parquet'))

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, legend=True, legend_label="Gold Standard - Impossible", legend_min=5.31, legend_max=12.75, shift_adjustment=0.02)

In [ ]:
df_targeted.item_name.value_counts().index.tolist()

In [ ]:
items_less_than_2_dollars = (
    df_targeted
    .groupby('item_name')
    ['unit_price']
    .max()
    .to_frame(name='max_unit_price')
    .query('max_unit_price < 200.0')
    .reset_index()
    #.query('~item_name.isin(["Pumpkin Bread","Muffin Marionberry Cobbler","Muffin Chocolate"])')
    .item_name
    .tolist()
    )
missed_merch = ['Hi','Cool']
missed_drinks = []
take_and_go = []

modification_name_changes = [
    
]

name_changes = {

}

filtered = (
    df_targeted
    .query('~item_name.isin(@items_less_than_2_dollars)')
    .query('~item_name.isin(@missed_merch)')
    .query('~item_name.isin(@missed_drinks)')
    .query('~item_name.isin(@take_and_go)')
    .pipe(lambda df: rename_items(df, name_changes))
    #.pipe(lambda df: rename_items_by_modifications(df, modification_name_changes))
)

print(
    filtered
    .groupby('item_name')
    ['item_modifications']
    .apply(lambda s: s.value_counts().index.str.slice(0,20).tolist()[0:3])
    .loc[filtered.item_name.value_counts().index.tolist()]
    .to_frame()
    .join(filtered.item_name.value_counts())
    .reset_index()
    .set_index('count')
    [['item_name','item_modifications']]
    #.iloc[10:,:]
    .to_string()
)

plot_dish_time_series(filtered.set_index('created_at'), loc_id, before_after_details_true, top_n=70)

In [ ]:
presence_dict = {}
for item, group in filtered.groupby("item_name"):
    presence_dict[item] = infer_active_days(group["created_at"], max_gap_days=120)
presence_df = pd.concat(presence_dict, axis=1).fillna(False)

presence_weekly = presence_df.resample('W').max().fillna(False).astype(bool).to_period('W')
dish_order = list(filtered.item_name.value_counts().index)
plot_boolean_time_series(presence_weekly, loc_id, before_after_details_true, dish_order, [])
plot_dish_time_series(filtered.set_index('created_at'), loc_id, before_after_details_true, top_n=70)

presence_daily = strict_bridge_fill(presence_df, limit=7).resample('D').max()
#presence_daily = (presence_weekly.astype(float).replace(0.0, np.nan)[vegetarian_dishes].resample('D').interpolate(limit=6).fillna(0).astype(int).sum(axis=1).plot())
menu = pd.read_csv(Path("scripts") / "labeling" / "dish_labels" / (loc_id + ".csv"))

vegan_dishes = menu.loc[menu["vegan"], "item_name"]
vegetarian_dishes = menu.loc[menu["vegetarian"], "item_name"]
mpbamod_dishes = menu.loc[menu["mpbamod"], "item_name"]

vegan_dishes_count = presence_daily[vegan_dishes].sum(axis=1)
vegetarian_dishes_count = presence_daily[vegetarian_dishes].sum(axis=1)
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
indicator = pd.to_datetime(promo_date) < presence_daily.index
mpbamod_dishes_count = presence_daily[mpbamod_dishes].sum(axis=1) * indicator
menu_counts = pd.concat([vegan_dishes_count, vegetarian_dishes_count, mpbamod_dishes_count], axis=1).set_axis(['vegan_dishes_count', 'vegetarian_dishes_count', 'mpbamod_dishes_count'], axis=1)
menu_counts.plot()
menu_counts.to_csv(Path("scripts") / "labeling" / "dish_counts" / (loc_id + ".csv"))

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
print(df_uncleaned.query('item_name.str.contains("Impossible")')['item_quantity'].sum())
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Impossible")')['item_quantity'].sum())

plot_time_series_subset(
    df_uncleaned, 
    exposure=promo_date,
    freq='W',
    truncate=True)
plt.show()
plot_time_series_subset(
    df_uncleaned.query('item_name.str.contains("Impossible")'),
    exposure=promo_date, 
    freq='W', 
    truncate=False)
plt.show()

In [ ]:
# Price analysis
print(df_consolidated
      .query('~vegetarian')
      ['unit_price']
      .mean())

print((df_consolidated
       .query('~vegetarian')
       ['item_name']
       .nunique()) / (df_consolidated
                      ['item_name']
                      .nunique()))
plt.plot(df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['unit_price']
         .mean(), 
         df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"').resample('W')['item_name'].nunique() / df_consolidated.query('dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['item_name']
         .nunique(), 
         'o', 
         alpha=0.5)
plt.show()